# Loan default prediction

Predicting mortgage loan default using the [Kaggle Loan Default Dataset](https://www.kaggle.com/datasets/yasserh/loan-default-dataset) (~148,670 loans). Covers data leakage detection, missing-value strategy, encoding, three model families (logistic regression, Random Forest, a small neural network), threshold tuning, and SHAP-based interpretability.


## Step 1: Load the data


In [ ]:
# Upload the CSV from Kaggle directly into the Colab session
from google.colab import files
uploaded = files.upload()


In [ ]:
import pandas as pd

# Load the dataset and take a first look at its shape and structure
df = pd.read_csv('Loan_Default.csv')
print(df.shape)
df.head()


In [ ]:
# Check column dtypes - tells us which columns are numeric vs categorical (object)
df.info()


In [ ]:
# Count missing values per column, worst first
df.isnull().sum().sort_values(ascending=False)


In [ ]:
# Same info as above, but as a percentage - easier to judge severity at a glance
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_pct.round(2)
})

missing_summary[missing_summary['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)


### Column reference

Grouped by category rather than alphabetically, for easier reasoning during feature engineering.

**Identifiers / admin**
- `ID`, row identifier, not predictive
- `year`, record year

**Loan structure**
- `loan_limit`, conforming ('cf') vs non-conforming loan
- `loan_type`, product type (type1/type2/type3)
- `loan_purpose`, coded purpose (p1-p4)
- `loan_amount`, amount borrowed
- `term`, duration in months
- `business_or_commercial`, personal vs business/commercial use

**Approval & underwriting**
- `approv_in_adv`, pre-approved ('nopre' vs 'pre')
- `Credit_Worthiness`, internal tier (l1, l2)
- `open_credit`, other open credit accounts
- `submission_of_application`, how the application was submitted

**Risk-related loan terms**
- `Neg_ammortization`, allows negative amortization (balance can grow, riskier)
- `interest_only`, interest-only period (no principal reduction, riskier)
- `lump_sum_payment`, balloon payment required at the end

**Property / collateral**
- `property_value`, `construction_type`, `occupancy_type`, `Secured_by`, `total_units`, `Security_Type`
- `LTV`, loan-to-value ratio = loan_amount / property_value (higher = riskier)

**Borrower financial profile**
- `income`, `credit_type`, `Credit_Score`, `co-applicant_credit_type`, `dtir1` (debt-to-income ratio), `age`, `Gender`

**Geography**
- `Region`

**Target**
- `Status`, 1 = defaulted, 0 = did not default

**Dropped for leakage**
- `rate_of_interest`, `Interest_rate_spread`, `Upfront_charges`, post-origination fields, missing almost exclusively for defaults (see below)


## Step 2: Confirm and understand the target variable


In [ ]:
# Confirm the target column's actual values (not just trusting the column name)
df['Status'].unique()


In [ ]:
df['Status'].value_counts()


In [ ]:
# Same counts, but as percentages - makes the class imbalance easy to read
df['Status'].value_counts(normalize=True) * 100


In [ ]:
# Quick gut-check: do defaulters look different on a few obvious features?
df.groupby('Status')[['loan_amount', 'income', 'Credit_Score']].mean()


### Reading the group means

- **loan_amount**: defaulters borrowed slightly *less* on average (319k vs 335k), small gap, not a strong standalone signal.
- **income**: defaulters have noticeably lower average income (~6,232 vs ~7,204), consistent with the idea that lower income relative to loan size leaves less of a cushion.
- **Credit_Score**: almost identical between the two groups (699.5 vs 700.6), surprising, since credit score is normally expected to be one of the strongest default predictors.

**Takeaway:** a variable's name doesn't guarantee predictive power. The near-zero gap on `Credit_Score` could mean the field isn't very informative in this dataset, or that its effect only shows up in combination with other features (something a simple group mean can't capture, but SHAP can, later on). No conclusions yet, just something to keep an eye on when it comes to feature importance.


In [ ]:
df['Credit_Score'].describe()


### Missing values: random, or meaningful?

The real question for each column with missing data is whether it's missing at random, or whether the *fact* it's missing carries information on its own.

`Interest_rate_spread` and `rate_of_interest` are missing in almost exactly the same count as `Status == 1` (36,639), worth checking directly rather than assuming it's a coincidence.


## Step 3: Check for data leakage in the missing values


In [ ]:
# Check whether missingness in these two columns lines up with defaults specifically
df.groupby('Status')[['Interest_rate_spread', 'rate_of_interest']].apply(lambda x: x.isnull().sum())


### A leakage signal

**What the numbers show:** every missing value in `Interest_rate_spread` and `rate_of_interest` occurs only when `Status == 1` (default). Non-defaulters have zero missing values in these columns, a near-perfect split, not a coincidence.

**What it likely means:** these fields are probably only populated once a loan is fully processed under normal terms. If a loan defaulted, that processing step may never have completed, so the field stayed empty. In other words, the missingness itself is encoding the target variable.

**Why that's a problem, not just an inconvenience:** this is a textbook case of *data leakage*. A model could learn "missing `rate_of_interest` → predict default" and score very well in testing, but a bank deciding on a *new* application has no way of knowing in advance whether these post-approval fields will end up missing, because the outcome hasn't happened yet. Training on this would mean training a model that's secretly already seen the answer.

**Options:**
- Drop the columns entirely (safest, since the missingness pattern is essentially leakage in disguise).
- Keep them only if it can be confirmed they're genuinely available before the outcome is known (not the case here).


In [ ]:
# NOTE: this line does NOT reassign df, so it's just a preview of the drop -
# the actual drop (including Upfront_charges) happens a few cells down.
df.drop(columns=['Interest_rate_spread', 'rate_of_interest'])


In [ ]:
df.groupby('Status')['Upfront_charges'].apply(lambda x: x.isnull().sum())


In [ ]:
# Convert to % missing within each class, since the classes are different sizes
df.groupby('Status')['Upfront_charges'].apply(lambda x: x.isnull().mean() * 100)


`Upfront_charges` is missing for almost every defaulted loan and present for almost every non-defaulted loan, the same suspicious imbalance as before. Same leakage logic applies: this field is likely only recorded once a loan is fully processed/funded, which may never happen for a defaulted loan.


In [ ]:
# Drop the three leakage columns for good - all three are missing almost
# exclusively for defaulted loans, so keeping them would leak the target
df = df.drop(columns=['Interest_rate_spread', 'rate_of_interest', 'Upfront_charges'])


In [ ]:
# Check the remaining suspicious columns the same way
df.groupby('Status')[['dtir1', 'property_value', 'LTV']].apply(lambda x: x.isnull().mean() * 100)


### A different pattern

`property_value` and `LTV` have identical missing percentages (41.2% / 0.0018%), not a coincidence, since **LTV is a ratio: `loan_amount / property_value`**. If `property_value` is missing, `LTV` can't be calculated either, so they're always missing together. That's a mathematical dependency, not leakage.

**Is this the same leakage problem as before?** Think about *when* each field would realistically be known:
- `rate_of_interest`, `Upfront_charges` → finalized after loan origination → missing ≈99% for defaults → leakage.
- `property_value`, `LTV`, `dtir1` → typically known at application time, before approval and long before anyone knows the outcome.

So while defaulters do show higher missingness here (44.5% vs 7.0% for `dtir1`; 41.2% vs ~0% for `property_value`/`LTV`), this looks like a different situation, riskier borrowers may genuinely submit messier or incomplete paperwork, and that itself could be a legitimate signal available at decision time rather than a leak from the future.


### Why timing, not correlation, is the deciding factor

The question worth answering: why is it okay to treat missingness as a signal for `property_value`/`LTV`/`dtir1`, but not for `rate_of_interest`?

`rate_of_interest` is tied to a stage that happens *after* the loan is underway, its absence is a byproduct of how the loan ended up, not something knowable at approval time. That's leakage.

`property_value` is different: it's not that it *can't* be missing (41% of defaulters are missing it), the real test is whether it *could have existed at application time, regardless of outcome*. A bank would ask for or estimate property value right at application. So even though it's often missing for defaulters, that missingness reflects something real and available at decision time (incomplete paperwork, a rushed application, a less thorough applicant), not a fact from the future.

**The dividing line:**
- `rate_of_interest` missing → caused by the outcome → leakage, drop it.
- `property_value` missing → exists independently of the outcome, at decision time → legitimate feature, even though it correlates with risk.

The concept to hold onto: **timing relative to the outcome**, not just whether something correlates with default.


## Step 4: Imputation strategy for the remaining missing values


In [ ]:
# Compare mean vs median (and max vs 75th percentile) to pick an imputation strategy
df[['dtir1', 'property_value', 'LTV', 'income']].describe()


### Choosing an imputation strategy: mean vs. median

Comparing `mean` against the `50%` (median) row, plus the `max` against the `75%` value as an outlier check:

- **dtir1**: mean 37.7, median 39.0, close (diff ~1.3). Min 5, max 61, no wild outliers. Either mean or median would work here.
- **income**: mean 6,957, median 5,760, mean noticeably higher. Max is 578,580 against a 75th percentile of just 8,520, a huge gap, meaning a small number of very high earners are pulling the mean up. Classic right-skew → **use median**.
- **property_value**: mean ≈497,893, median ≈418,000, again mean > median by a wide margin. Max is 16,508,000 against a 75th percentile of 628,000, a massive outlier gap. Strongly right-skewed → **use median**.
- **LTV**: mean 72.7, median 75.1, reasonably close, but the max (7,831) is almost certainly a data error rather than a real loan-to-value ratio, given the 25%-86% range for most of the data. Use median, but flag this column for possible outlier capping later.

**General rule:** whenever mean and median diverge, or the max dwarfs the 75th percentile, that signals skew or outliers, median imputation is the safer default since it isn't dragged around by extreme values.


### Missingness flags before imputing

`property_value` (and by extension `LTV`) is missing in 41.2% of defaults vs. ~0% of non-defaults, too strong a pattern to quietly fill in and lose. Once median imputation happens, that signal disappears unless it's explicitly preserved as its own feature first.


In [ ]:
# Step 1: create "was this missing" flags BEFORE filling anything in,
# since these columns are legitimately missing at application time (not leakage)
for col in ['dtir1', 'property_value', 'LTV', 'income']:
    df[col + '_missing'] = df[col].isnull().astype(int)


In [ ]:
# Step 2: now impute with the median (robust to the skew/outliers seen above)
for col in ['dtir1', 'property_value', 'LTV', 'income']:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)


**Why order matters:** imputing first means `isnull()` returns all `False` afterward, the missingness information is lost for good. The flag has to be created *before* the values are filled.

**Result:** four new binary columns (`dtir1_missing`, `property_value_missing`, `LTV_missing`, `income_missing`) preserve the "was this originally missing" signal, while the original columns are now fully filled with median values ready for modeling.

Verification below should show 0 missing values across all four columns.


In [ ]:
# Verify: all four columns should now show 0 missing values
df[['dtir1', 'property_value', 'LTV', 'income']].isnull().sum()


## Step 5: Handle the small remaining missing-value columns


In [ ]:
# Inspect the small remaining missing-value columns: dtype + value counts
# (dropna=False keeps NaN visible as its own category in the count)
for col in ['loan_limit', 'approv_in_adv', 'age', 'submission_of_application',
            'loan_purpose', 'Neg_ammortization', 'term']:
    print(col, '-', df[col].dtype)
    print(df[col].value_counts(dropna=False).head())
    print()


### Handling the remaining small missing-value columns

Checking dtype and value counts together rather than filling blindly:
- Most of these are categorical (`loan_limit`, `approv_in_adv`, `age`, `submission_of_application`, `loan_purpose`, `Neg_ammortization`), standard move is **mode imputation** (fill with the most frequent category), since median doesn't apply to non-numeric data.
- `term` is numeric (loan duration in months), median imputation like before.

Since all of these are under 1% missing, the exact method matters far less than it did for `income` or `property_value`, with so little missing data, almost any reasonable choice has a negligible effect. Worth saving careful reasoning for the columns with substantial missingness, and not overthinking the ones that are nearly complete.


Clean pattern across the board, each column has one dominant category, which is exactly when mode imputation makes sense (filling with "whatever is most common" barely changes the data's overall shape).

**Note:** `age` didn't show a `NaN` row in the output above, even though the earlier missing-values count showed 200 missing for `age`. That's because `.head()` only shows the top 5 rows, and `NaN` wasn't among the top 5 most frequent categories, it got cut off from view. Lesson: `.head()` can hide things; absence in a preview doesn't mean zero.


In [ ]:
# Mode imputation for the categorical columns - fill with the most frequent value
categorical_cols = ['loan_limit', 'approv_in_adv', 'age', 'submission_of_application',
                     'loan_purpose', 'Neg_ammortization']

for col in categorical_cols:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)


`term` is numeric, but 121,685 of ~148,670 rows (about 82%) are exactly 360.0, this isn't a "spread out" numeric column, it's dominated by one value, much like the categoricals. Mode imputation makes more sense than median here too.


In [ ]:
# term is numeric but dominated by one value (360.0 in ~82% of rows),
# so mode imputation fits better here than median
term_mode = df['term'].mode()[0]
df['term'] = df['term'].fillna(term_mode)


In [ ]:
# Final check: total missing values across the whole dataframe should be 0
df.isnull().sum().sum()


## Step 6: Encode categorical variables


In [ ]:
# List every remaining categorical (object dtype) column
df.select_dtypes(include='object').columns


In [ ]:
# Check cardinality (unique value count) per categorical column -
# this decides one-hot vs ordinal encoding
for col in df.select_dtypes(include='object').columns:
    print(col, '-', df[col].nunique())


### Why cardinality matters for encoding choice

- **Low cardinality (2-5 categories)** → one-hot encoding is fine (e.g. `Gender` creates just 2-3 new columns).
- **Ordinal (has a natural order)** → `age` (25-34, 35-44, ...) shouldn't be one-hot encoded, since that would throw away the real ranking. Better mapped to ordered integers (0, 1, 2, 3...).
- **High cardinality** → a column with 50+ unique values would explode into 50+ mostly-sparse one-hot columns, hurting performance and interpretability. Not expected to be an issue here, since most categorical fields in this dataset look like small fixed sets (e.g. 'type1/type2/type3').


Every column here is low cardinality, the max is `age` with 7 categories, so no high-cardinality problem to work around.

**Ordinal (needs manual ordering), one column:**
- `age` (25-34, 35-44, 45-54, 55-64, 65-74, plus two more brackets) → map to ordered integers, don't one-hot this one.

**Everything else (20 columns), no natural order, one-hot encode:**
`loan_limit`, `Gender`, `approv_in_adv`, `loan_type`, `loan_purpose`, `Credit_Worthiness`, `open_credit`, `business_or_commercial`, `Neg_ammortization`, `interest_only`, `lump_sum_payment`, `construction_type`, `occupancy_type`, `Secured_by`, `total_units`, `credit_type`, `co-applicant_credit_type`, `submission_of_application`, `Region`, `Security_Type`


In [ ]:
# Step 1 - inspect age's categories before hand-coding the order
# (Python's default sort puts '<25' and '>74' in the wrong spot alphabetically)
print(sorted(df['age'].unique()))


In [ ]:
# Manually define the real order of the age brackets
age_order = ['<25', '25-34', '35-44', '45-54', '55-64', '65-74', '>74']


In [ ]:
# Build a {category: rank} dictionary and map it onto the column,
# turning age into an ordinal (ordered) numeric feature
age_mapping = {age: i for i, age in enumerate(age_order)}
df['age'] = df['age'].map(age_mapping)


In [ ]:
# Confirm age is now numeric (0-6) instead of text brackets
df['age'].unique()


In [ ]:
# Drop_first=True avoids the "dummy variable trap" - one category per
# column is implied by all the others being 0, so it's left out to avoid redundancy
onehot_cols = ['loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose',
               'Credit_Worthiness', 'open_credit', 'business_or_commercial',
               'Neg_ammortization', 'interest_only', 'lump_sum_payment',
               'construction_type', 'occupancy_type', 'Secured_by', 'total_units',
               'credit_type', 'co-applicant_credit_type', 'submission_of_application',
               'Region', 'Security_Type']

df = pd.get_dummies(df, columns=onehot_cols, drop_first=True)


In [ ]:
df.shape


In [ ]:
# Confirm no 'object' dtype columns remain - everything should be numeric/bool now
df.dtypes.value_counts()


In [ ]:
df.head()


47 columns total now, and critically, **zero `object` dtype columns remaining**, everything is `bool` (32, the one-hot columns), `int64` (10), or `float64` (5). Every categorical column has been successfully converted to something numeric, which is exactly what a model needs.

**Note for later:** scikit-learn generally handles `bool` columns fine, but if a dtype error ever comes up, `df = df.astype({col: 'int' for col in df.select_dtypes('bool').columns})` converts them all to 0/1 integers.


## Step 7: Train/test split


In [ ]:
from sklearn.model_selection import train_test_split

# Features: everything except the target (Status) and the row identifier (ID),
# since ID carries no real predictive signal
X = df.drop(columns=['Status', 'ID'])
y = df['Status']

# stratify=y keeps the same ~75/25 class balance in both the train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
X_train.shape, X_test.shape


In [ ]:
# Verify stratify=y worked: both should show roughly the same ~75/25 split
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)


### What `stratify=y` does

Without it, `train_test_split` shuffles all 148,670 rows randomly and cuts 80/20. Purely by chance, that could put a slightly different ratio of defaulters into train vs. test, not a huge shift, but with class imbalance already a concern, it's an avoidable source of noise.

`stratify=y` preserves the same class proportion in both splits, since the full dataset is ~75.4% class 0 / 24.6% class 1, both `y_train` and `y_test` independently keep that same ~75.4%/24.6% split. This removes one source of randomness, so any performance difference observed later comes from the model, not an accidental imbalance in the split.

`value_counts(normalize=True)` on `y_train`/`y_test` is just a verification step, proving that `stratify=y` actually delivered on that promise.


## Step 8: Baseline model - Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression

# Baseline logistic regression - max_iter raised above the sklearn default (100)
# to give the optimizer more room to converge
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)


### Understanding the ConvergenceWarning

**What "failed to converge" means:** logistic regression finds its coefficients through iterative optimization, small steps trying to minimize error, expected to "settle" on stable values. This warning means it hit the 1000-iteration cap before settling.

**Why it's happening:** `income` ranges up to ~578,580, `property_value` up to 16,508,000, while the one-hot columns are just 0s and 1s and `age` is 0-6. That's an enormous scale mismatch, the optimizer is sensitive to this, since features with huge numeric ranges dominate the math and make the optimization surface harder to navigate efficiently.

**The fix:** feature scaling was skipped up to this point (categoricals needed encoding first). Tree-based models like Random Forest don't need this, but linear models like logistic regression do.


In [41]:
# Fix: scale the numeric features:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### `scaler.fit_transform(X_train)`, what's actually happening

Two things at once:
- **fit**, the scaler looks at `X_train` and learns, per column, the average value and the spread (standard deviation).
- **transform**, using what it just learned, it rescales every column onto a comparable footing, roughly centered around 0, regardless of whether the original column was `age` (0-6) or `property_value` (up to 16 million).

**Why it matters:** without scaling, a feature like income (tens of thousands) would dominate the math over a feature like age (single digits), even if age matters just as much for the outcome. Scaling puts every feature on equal footing so the model judges each one by its actual relationship to the target, not its raw size.

`scaler.transform(X_test)`, no `fit` here, only `transform`. The exact scaling rules learned from training data get applied to the test set; the scaler never "peeks" at test data to learn new rules. That would be a form of leakage, similar to the `rate_of_interest` issue earlier.


**Why `fit_transform` on train but only `transform` on test:** the scaler learns mean/standard deviation from training data only, then applies that same learned scale to test data. Fitting on test data too would let information from the test set leak into preprocessing, a subtler version of the same leakage principle covered earlier, just applied to scaling instead of raw features.


In [42]:
# Now retrain on the scaled data:

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

### What `max_iter` controls

It caps how many optimization steps logistic regression can take before being forced to stop, whether or not it's found the best solution yet.

**Default is 100** if unspecified. The first run here used `max_iter=1000`, and even that wasn't enough given the unscaled data, the default of 100 would have triggered the warning even sooner.

**If training gets cut off early:** the model doesn't crash, it still returns a fitted object and makes predictions, but those predictions may rest on coefficients that hadn't fully settled, which can mean slightly worse, less reliable results. That's why it's a warning and not an error.

**Why 1000:** no deep science, a generous "give it room" value. For a dataset this size, 1000 iterations is more than enough once scaling is applied properly.

**Habit worth keeping:** seeing a `ConvergenceWarning` should prompt "is my data scaled?" before reaching for a higher `max_iter`, throwing more iterations at unscaled data usually just delays the same underlying problem.


In [ ]:
# Evaluate on the test set using metrics that actually matter for imbalanced classes
# (accuracy alone would be misleading given the ~75/25 split)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


### Reading the confusion matrix

```
                    Predicted: No Default    Predicted: Default
Actual: No Default        22,188 (TN)             218 (FP)
Actual: Default             3,672 (FN)           3,656 (TP)
```

- **True Negatives (22,188):** correctly predicted "no default"
- **False Positives (218):** predicted "will default," but didn't
- **False Negatives (3,672):** predicted "no default," but actually did default ⚠️
- **True Positives (3,656):** correctly caught actual defaulters

**Accuracy (86.9%)** looks strong, but it's inflated by class imbalance, even a lazy "always predict no default" model would score ~75%.

**Precision (94.4%)**, of everyone flagged as "will default," 94.4% actually did. Very few false alarms.

**Recall (49.9%)**, the number that matters most here. Of all 7,328 real defaulters in the test set, only half were caught; the other half slipped through as "safe" predictions.

**Why this matters practically:** a bank's biggest risk is approving a loan for someone who defaults. A missed defaulter (false negative) usually costs real money; a false alarm just means extra scrutiny on someone who was fine. Missing half of all actual defaulters is a serious weakness, even with an accuracy that "looks" good.

**F1 (0.65)** is the harmonic average of precision and recall, pulled down specifically because recall is so much weaker than precision.

**ROC-AUC (0.74)** measures separation across all thresholds, not just 0.5, decent, but with room to improve.

**Why this happens:** by default, logistic regression only predicts "default" when more than 50% confident. With only 24.6% of training data being defaulters, the model has naturally learned to be cautious about the minority class.

**Two fixes to try:**
1. `class_weight='balanced'`, penalize mistakes on defaulters more heavily during training.
2. Lower the decision threshold, e.g. predict "default" above 0.35 instead of 0.5, catching more true defaulters at the cost of more false positives.


### Recall, precision, F1, ROC-AUC, definitions with this project's numbers

**Recall**
- Answers: of everyone who *actually* defaulted, how many did the model catch?
- Formula: `TP / (TP + FN)`
- Here: `3,656 / (3,656 + 3,672)` = 49.9%
- There were 7,328 real defaulters in the test set; the model correctly flagged 3,656 and missed 3,672. Recall is about not letting the bad cases slip through, critical in settings like fraud detection, cancer screening, or default risk, where a missed positive case is costly.

**Precision**
- Answers: of everyone the model *flagged*, how many actually defaulted?
- Formula: `TP / (TP + FP)`
- Here: `3,656 / (3,656 + 218)` = 94.4%
- The model raised the alarm on 3,874 people, 94.4% of whom really were defaulters, very few false alarms. Precision is about not crying wolf too often.

**The tension:** these two trade off against each other. A cautious model (high precision, few false alarms) misses more real defaulters (low recall); a more aggressive model catches more defaulters but raises more false alarms.

**F1 score**
- Answers: a single number balancing precision and recall.
- It's the *harmonic mean* of the two (not a simple average), it punishes a big gap between them more than an average would. A plain average of 94.4% and 49.9% would be ~72%, but F1 comes out to 65.3%, exposing the imbalance rather than hiding it.
- Useful whenever both false alarms and missed cases matter, and a single number is needed to compare models.

**ROC-AUC**
- The model doesn't just output 0/1, internally it computes a probability, and a cutoff (usually 50%) turns that into a final decision. Moving that cutoff trades recall against precision.
- ROC-AUC asks how well the model separates the two classes across *every* possible cutoff, by plotting the true positive rate against the false positive rate and measuring the area under that curve.
- Scale: 0.5 = no better than random guessing, 1.0 = perfect separation. Here: 0.74, meaningfully better than random, but not great.
- Interpretation: given one random defaulter and one random non-defaulter, there's roughly a 74% chance the model ranks the defaulter as higher risk.
- Useful alongside F1 because F1 evaluates one specific threshold, while ROC-AUC captures the model's overall ranking ability, independent of wherever the cutoff ends up being set.


In [ ]:
# Scikit-learn's built-in confusion matrix plot - quick and simple
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No Default', 'Default'], cmap='Blues')


In [ ]:
# Seaborn heatmap version - more customizable
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Logistic Regression Baseline')
plt.show()


### `class_weight='balanced'`

Tells logistic regression to weight the minority class (defaulters) more heavily during training, penalizing mistakes on them more since they're underrepresented.


In [ ]:
# Retrain with class_weight='balanced' - penalizes mistakes on the minority
# class (defaulters) more heavily, since they're underrepresented
model_balanced = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = model_balanced.predict(X_test_scaled)


**What it does under the hood:** normally every training example counts equally in the loss function. With `class_weight='balanced'`, scikit-learn calculates weights inversely proportional to class frequency, since defaulters are ~24.6% of the data (rarer), each defaulter example gets weighted more heavily. The model gets penalized harder for misclassifying a defaulter, pushing it to be less conservative about predicting the minority class.


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_balanced))
print("Precision:", precision_score(y_test, y_pred_balanced))
print("Recall:", recall_score(y_test, y_pred_balanced))
print("F1:", f1_score(y_test, y_pred_balanced))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_balanced))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_balanced))


In [ ]:
cm = confusion_matrix(y_test, y_pred_balanced)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Balanced Logistic Regression')
plt.show()


In [ ]:
# Side-by-side comparison: baseline vs balanced weights
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_baseline = confusion_matrix(y_test, y_pred)
cm_balanced = confusion_matrix(y_test, y_pred_balanced)

sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'], ax=axes[0])
axes[0].set_title('Baseline')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_balanced, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'], ax=axes[1])
axes[1].set_title('Balanced (class_weight)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()


**Worth reasoning through:** 2,499 false positives means 2,499 safe borrowers get flagged for extra scrutiny (manual review, a slightly higher rate, more documentation), in exchange for catching 1,148 more real defaulters. There's no universally "correct" answer here; it depends on what a false positive actually costs the bank versus what a false negative costs. Business judgment as much as a technical one.


## Step 10: Threshold tuning

Instead of letting the model's default 50% cutoff decide "default vs. no default," the actual probabilities the model produces can be used directly, choosing manually where to draw the line.

First, get probabilities instead of hard predictions:


In [ ]:
# Get raw probabilities instead of hard 0/1 predictions -
# [:, 1] selects the probability of class 1 (default)
y_proba = model.predict_proba(X_test_scaled)[:, 1]


`.predict()` (used earlier) gives 0s and 1s directly, it already applied the 50% cutoff internally. `.predict_proba()` instead returns the raw probability of each class for every row, a number between 0 and 1. `[:, 1]` takes just the second column, the probability of class 1 (default).


In [ ]:
y_proba[:10]


Output looks like `[0.12, 0.63, 0.08, 0.45, ...]`, the model's actual confidence scores, before any cutoff is applied.

Now try a lower threshold than the default 0.5, say 0.35:


In [ ]:
# Apply a lower threshold than the default 0.5, to catch more real defaulters
threshold = 0.35
y_pred_thresh = (y_proba >= threshold).astype(int)


`y_proba >= threshold` compares every probability against 0.35 and returns `True`/`False` per row. `.astype(int)` converts those into 1s and 0s. Instead of requiring >50% confidence before flagging someone as a default risk, this flags anyone at ≥35%, a lower bar that should raise recall (catch more real defaulters) at the cost of some precision (more false alarms).


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_thresh))
print("Precision:", precision_score(y_test, y_pred_thresh))
print("Recall:", recall_score(y_test, y_pred_thresh))
print("F1:", f1_score(y_test, y_pred_thresh))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_thresh))


In [ ]:
cm_thresh = confusion_matrix(y_test, y_pred_thresh)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_thresh, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Threshold 0.35')
plt.show()


In [ ]:
# Three-way comparison: baseline, balanced weights, threshold 0.35
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cms = [
    (confusion_matrix(y_test, y_pred), 'Baseline (0.5 cutoff)'),
    (confusion_matrix(y_test, y_pred_balanced), 'Balanced weights'),
    (confusion_matrix(y_test, y_pred_thresh), 'Threshold 0.35')
]

for ax, (cm, title) in zip(axes, cms):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Default', 'Default'],
                yticklabels=['No Default', 'Default'], ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


### Threshold 0.35 vs. the other two

| Metric | Baseline (0.5) | Balanced weights | Threshold 0.35 |
|---|---|---|---|
| Precision | 94.4% | 65.8% | 83.0% |
| Recall | 49.9% | 65.6% | 56.0% |
| F1 | 65.3% | 65.7% | 66.9% |

Threshold 0.35 lands the best F1 of the three so far, and gives a middle-ground trade-off: better recall than baseline, better precision than the balanced-weight model. This is the kind of control threshold tuning provides, not stuck picking between two extremes, the threshold can be dialed to whatever trade-off fits the business need.


### How the model actually turns a probability into a decision

Internally, the model computes a probability, e.g. "63% confident this person will default." Somewhere that probability has to become a yes/no decision; scikit-learn defaults to 50% as the cutoff (**the threshold**). That's a rule imposed on top of the probability, not something the model itself demands.

**Why move it:** with the 50% bar, the model missed half of all real defaulters, many of them probably scored 35-45%, genuinely elevated risk that just didn't clear the bar. Threshold tuning means deliberately lowering (or raising) that bar to change what gets flagged.

Lowering to 35% catches people who were previously just under the old bar, which is why recall went up in that experiment, but also flags some genuinely safe people whose risk score was only 35-40%, which is why precision dropped a bit.

**Key idea:** nothing about the model changes. The probabilities are identical either way, only the confidence level required to act on them changes.

**Analogy:** a very sensitive smoke detector (low threshold) never misses a real fire but false-alarms on burnt toast; a less sensitive one (high threshold) has fewer false alarms but might miss a real fire until it's serious. Neither setting is "wrong", it depends on the relative cost of a missed fire versus a false alarm, exactly the decision being made when picking a threshold for loan defaults.

Testing a single threshold (0.35) is like testing one smoke-detector sensitivity and stopping. The more thorough approach is sweeping a full range (0.1 to 0.9) and watching how precision, recall, and F1 all shift, then picking the point that best matches the real-world cost trade-off.


In [ ]:
# Confirm which column of predict_proba() actually corresponds to class 1 -
# never assume, always check model.classes_
model.classes_        # confirms order: [0, 1]
model.predict_proba(X_test_scaled)[:5]   # first 5 rows, both columns
y_proba[:5]            # confirms this matches column index 1 specifically


In [ ]:
# Sweep a full range of thresholds instead of testing one at a time,
# to see the whole precision/recall/F1 trade-off at once
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    precisions.append(precision_score(y_test, preds))
    recalls.append(recall_score(y_test, preds))
    f1s.append(f1_score(y_test, preds))

precisions = np.array(precisions)
recalls = np.array(recalls)
f1s = np.array(f1s)

# Find where precision and recall are closest (their "crossing point")
diff = np.abs(precisions - recalls)
cross_idx = np.argmin(diff)
cross_threshold = thresholds[cross_idx]

# Find where F1 peaks
best_f1_idx = np.argmax(f1s)
best_f1_threshold = thresholds[best_f1_idx]

plt.figure(figsize=(9, 6))
plt.plot(thresholds, precisions, label='Precision', marker='o')
plt.plot(thresholds, recalls, label='Recall', marker='o')
plt.plot(thresholds, f1s, label='F1', marker='o')

# Mark the precision/recall crossing point
plt.scatter(cross_threshold, precisions[cross_idx], color='red', zorder=5, s=100)
plt.annotate(f'P≈R at {cross_threshold:.2f}',
             (cross_threshold, precisions[cross_idx]),
             textcoords="offset points", xytext=(10, 10), fontsize=9, color='red')

# Mark the best F1 point
plt.scatter(best_f1_threshold, f1s[best_f1_idx], color='green', zorder=5, s=100)
plt.annotate(f'Best F1={f1s[best_f1_idx]:.2f} at {best_f1_threshold:.2f}',
             (best_f1_threshold, f1s[best_f1_idx]),
             textcoords="offset points", xytext=(10, -15), fontsize=9, color='green')

plt.xlabel('Threshold')
plt.ylabel('Score')
plt.legend()
plt.title('Precision / Recall / F1 vs Threshold')
plt.grid(alpha=0.3)
plt.show()

print(f"Precision ≈ Recall around threshold: {cross_threshold:.2f}")
print(f"Best F1 ({f1s[best_f1_idx]:.3f}) occurs at threshold: {best_f1_threshold:.2f}")


The point where precision and recall are most balanced (and/or F1 peaks) is threshold **0.25**, lower than both 0.5 and 0.35.

**What this means in practice:** the model's probability estimates for defaulters tend to sit lower than expected, a lot of genuine defaulters score in the 0.25-0.35 range, not clearing the higher bars tried before. Dropping the threshold to 0.25 catches more of them.


In [ ]:
# Final chosen threshold based on the sweep above
threshold_final = 0.25
y_pred_final = (y_proba >= threshold_final).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall:", recall_score(y_test, y_pred_final))
print("F1:", f1_score(y_test, y_pred_final))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_final))


In [ ]:
cm_final = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Threshold 0.25')
plt.show()


## Locking in a logistic regression configuration

| Model | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|
| Baseline (0.5) | 94.4% | 49.9% | 65.3% | 0.745 |
| Balanced weights | 65.8% | 65.6% | 65.7% | 0.772 |
| Threshold 0.35 | 83.0% | 56.0% | 66.9% |, |
| Threshold 0.25 | 66.3% | 65.1% | 65.7% |, |

**Verdict:** threshold 0.35 gives the best F1 (66.9%) while keeping precision reasonably high (83%), a solid middle ground. Threshold 0.25 and balanced weights land in nearly the same spot as each other (~65-66% on both precision and recall), since both push the model to be more aggressive about flagging risk.

**Chosen configuration: logistic regression with threshold = 0.35.**


## Step 11: Random Forest, a different model family

Unlike logistic regression, Random Forest doesn't need scaled data, it works directly on the original `X_train`/`X_test` splits, since tree splits work on raw thresholds rather than gradient-based optimization.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest doesn't need scaled features - trees split on raw thresholds,
# not gradient-based optimization
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))


### Random Forest vs. logistic regression

| Metric | LogReg baseline (0.5) | LogReg threshold 0.35 | Random Forest |
|---|---|---|---|
| Accuracy | 86.9% | 86.3% | 89.1% |
| Precision | 94.4% | 83.0% | 94.9% |
| Recall | 49.9% | 56.0% | 59.0% |
| F1 | 65.3% | 66.9% | 72.7% |
| ROC-AUC | 0.745 |, | 0.790 |

This is a genuinely better model across almost every metric, not just a trade-off shift like the threshold experiments were. Random Forest improved recall *and* kept precision high *and* improved ROC-AUC, which suggests it's capturing the underlying patterns better overall, likely because it can model non-linear relationships and feature interactions (e.g. "high debt-to-income ratio combined with a missing property value" as a specific risky combination) that a purely linear model structurally can't represent.

Recall is still only 59%, better than logistic regression's best, but still missing about 41% of real defaulters (3,007 false negatives). Switching model type helped more than any threshold tuning did, but the underlying challenge (a genuinely hard, imbalanced classification problem) hasn't disappeared. The same threshold-tuning approach could be applied to Random Forest's `predict_proba()` output to push recall further if needed.


In [ ]:
!pip install shap -q
import shap


In [ ]:
# Check the forest's size/depth - unrestricted max_depth means fully grown,
# potentially very deep trees, which makes SHAP noticeably slower to compute
print(rf_model.get_params()['max_depth'])
print(rf_model.n_estimators)


In [ ]:
# SHAP is expensive to compute, so explain a small random sample of the
# test set rather than the full ~30,000 rows
X_sample = X_test.sample(50, random_state=42)


In [ ]:
# TreeExplainer is optimized specifically for tree-based models like Random Forest
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_sample)


In [ ]:
# Check the shape of the output first - SHAP's return format can vary by version
import numpy as np
print(type(shap_values))
if isinstance(shap_values, list):
    print(len(shap_values))
    print(shap_values[0].shape)
else:
    print(shap_values.shape)


`shap_values` came back as a **50 samples × 46 features × 2 classes** array, the newer SHAP format, one single 3D array with the class dimension last (rather than a list of two arrays, one per class).

Since class 1 (default) is the outcome of interest, that slice needs to be pulled out separately:


In [ ]:
# Keep only the class-1 (default) slice: rows x features
shap_values_default = shap_values[:, :, 1]


This keeps all 50 rows and 46 features, but only the slice for class index 1 (default). `shap_values_default` is now a clean 50×46 array, one SHAP value per feature, per sample, specifically explaining how much that feature pushed toward predicting default.

Summary plot:


In [ ]:
shap.summary_plot(shap_values_default, X_sample)


**How to read the summary plot:**
- Features ranked top-to-bottom by overall importance (most influential at top)
- Each dot = one applicant from the sample
- Dot color = that applicant's actual value for the feature (red = high, blue = low)
- Dot position on the x-axis = how much that feature pushed *this specific* prediction toward "default" (right) or "no default" (left)


### Reading the SHAP summary plot

**Top finding, the missingness flags dominate.** `LTV_missing`, `property_value_missing`, and `dtir1_missing` are the top 3 most important features. The pattern is clear: red dots (value = 1, "this was missing") cluster to the right (pushes toward default), blue dots (value = 0, "not missing") cluster to the left.

This validates the earlier decision to keep these fields as engineered flags instead of dropping them (unlike `rate_of_interest`, which *was* dropped), the fact that this data was missing turned out to be one of the strongest legitimate signals in the whole dataset. Median-imputing without the flags would have thrown away the model's most powerful features.

**`dtir1` and `LTV` themselves** (the actual values, not the missing-flags) cluster tightly near zero with a slight blue-left/pink-right spread, meaningful, but far less powerful than their own missingness flags. Whether the data exists at all mattered more than what the actual number was.

**`income`:** clear pattern, low income (blue) pushes right toward default, high income (red) clusters left toward no default. Matches the intuition from the very first group-means check.

**`Credit_Score`:** notably small spread, dots tightly bunched near zero, confirms the earlier finding that credit score is barely influencing this model's predictions, despite being the feature most people would expect to dominate. A genuinely interesting, non-obvious result worth highlighting in any writeup.

**Caveat worth stating explicitly:** the missingness flags being this dominant is powerful, but also worth scrutiny, it's close to the same "near-perfectly-separates-the-classes" pattern that flagged `rate_of_interest` as leakage earlier. The difference is timing: this data would genuinely be knowable (or knowably missing) at application time, unlike `rate_of_interest`. Still, this is a strong predictor precisely because it's a strong *proxy* for something else in the application process (likely incomplete or rushed applications), not necessarily a fully understood causal driver.


In [ ]:
shap.dependence_plot('income', shap_values_default, X_sample)


### Dependence plot, going deeper on one feature

The summary plot shows that income matters and gives a rough direction (low income → higher default risk). A dependence plot goes further: it plots the actual income value on the x-axis against its SHAP contribution on the y-axis, for every applicant in the sample, showing the *shape* of the relationship (straight line? levels off? sharp cutoff?), not just its direction.

SHAP also auto-colors the dots by whichever feature interacts most strongly with income, a bonus signal showing which other feature changes how income affects risk.


### Reading the income dependence plot

**Shape:** as income rises from ~2,000 to ~6,000, the SHAP value drops sharply, from around +0.08 to around -0.04. From roughly 6,000 to 16,000+, the line flattens and stays around -0.02 to -0.06, with no further steep decline.

**In plain terms:** income matters a lot at the low end, moving from very low to moderate income substantially reduces predicted default risk. Past roughly the $6-7k mark, additional income barely matters, someone earning 16,000 isn't meaningfully "safer" in the model's eyes than someone earning 8,000. This is a diminishing-returns curve, not a straight line.

**Why that makes sense:** someone earning very little relative to their loan obligations is genuinely financially stretched, so every increment of income meaningfully improves their ability to repay. Once comfortably above a threshold, they're no longer the applicants at meaningful risk, so more income stops "buying" additional safety. A linear model's single coefficient could only express "higher income = lower risk" as a straight line, it couldn't capture this plateau shape.

**Interaction with `dtir1_missing`:** SHAP picked this as the feature most correlated with income's effect. The low-income, high-SHAP points near the top left (income ~1,000-3,000, SHAP +0.04 to +0.08) include several points where `dtir1_missing = 1`, hinting that low income combined with missing debt-to-income data is a particularly strong risk signal, stronger than either alone. This kind of interaction effect is something tree-based models can capture and linear models generally can't.

**Summary for writeup:** income shows a non-linear relationship with default risk, sharply protective at low income levels, with diminishing returns above ~$6-7k, and evidence of interaction with data-completeness features like `dtir1_missing`.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers


In [ ]:
# Simple feed-forward network for tabular binary classification:
# two shrinking hidden layers with dropout to reduce overfitting,
# sigmoid output for a 0-1 default probability
nn_model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])


In [ ]:
# binary_crossentropy is the standard loss for binary classification;
# adam is a reliable general-purpose optimizer
nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# class_weight rebalances training toward the minority class (default = 1),
# similar in spirit to class_weight='balanced' used for the other models
history = nn_model.fit(
    X_train_scaled, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    class_weight={0: 1, 1: 3}
)


In [ ]:
# Evaluate on the test set with the same metrics used for the other two models
y_pred_nn_proba = nn_model.predict(X_test_scaled)
y_pred_nn = (y_pred_nn_proba >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Precision:", precision_score(y_test, y_pred_nn))
print("Recall:", recall_score(y_test, y_pred_nn))
print("F1:", f1_score(y_test, y_pred_nn))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_nn))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nn))


### Neural network vs. the other two models

| Metric | LogReg (threshold 0.35) | Random Forest | Neural Network |
|---|---|---|---|
| Accuracy | 86.3% | 89.1% | 86.2% |
| Precision | 83.0% | 94.9% | 71.3% |
| Recall | 56.0% | 59.0% | 73.4% |
| F1 | 66.9% | 72.7% | 72.4% |
| ROC-AUC |, | 0.790 | 0.819 |

The neural network posted the best recall (73.4%, catching notably more real defaulters than either other model) and the best ROC-AUC (0.819, the best overall risk-ranking ability). F1 is essentially tied with Random Forest (72.4% vs. 72.7%). Random Forest still wins clearly on precision and accuracy.

This is a genuinely useful, honest result, tabular data usually favors tree ensembles as a general pattern, but that's not a fixed rule, and this is a concrete counterexample worth noting rather than glossing over. Some possible explanations:
- The neural net's `class_weight={0:1, 1:3}` was a more aggressive rebalancing than Random Forest's `class_weight='balanced'`, which alone could explain much of the recall/precision difference (same pattern seen earlier with logistic regression thresholds).
- The missingness-flag features and their interactions (which SHAP showed were powerful) may be exactly the kind of non-linear signal a neural network's hidden layers can also pick up on, not just tree splits.
- Architecture/training choices (dropout, epochs, batch size) all influence this result, a different configuration could shift these numbers in either direction. Worth noting as a limitation rather than a settled conclusion.

**Conclusion:** neither Random Forest nor the neural network is uniformly better. Random Forest offers higher precision and accuracy; the neural network achieves better recall and ROC-AUC. Model choice ultimately depends on the lender's actual cost trade-off between false positives and false negatives, not a single declared "winner."
